In [1]:
import os

import numpy as np
import pandas as pd

import librosa
import soundfile as sf
import warnings

from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from pathlib import Path
from transformers import ASTModel, ASTFeatureExtractor,  get_cosine_schedule_with_warmup
from tqdm import tqdm


warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)


In [2]:
texonomy_csv = pd.read_csv("/kaggle/input/competitions/birdclef-2026/taxonomy.csv")
train_csv = pd.read_csv("/kaggle/input/competitions/birdclef-2026/train.csv")

labels = pd.read_csv("/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv")
labels["numLabel"]=  labels["primary_label"].apply(lambda x: len(x.split(";")))

sampleSub = pd.read_csv("/kaggle/input/competitions/birdclef-2026/sample_submission.csv")

display(texonomy_csv.head())
print()
display(train_csv.head())
print()
display(labels.head())

,primary_label,inat_taxon_id,scientific_name,common_name,class_name
0,1161364,1161364,Guyalna cuta,Guyalna cuta,Insecta
1,116570,116570,Caiman yacare,Southern Spectacled Caiman,Reptilia
2,1176823,1176823,Leptodactylus luctator,Wrestler Frog,Amphibia
3,1491113,1491113,Adenomera guarani,Guaraní leaf-litter frog,Amphibia
4,1595929,1595929,Lysapsus limellum,Uruguay Harlequin Frog,Amphibia


,primary_label,secondary_labels,type,latitude,longitude,scientific_name,common_name,class_name,inat_taxon_id,author,license,rating,url,filename,collection
0,1161364,[],[],-22.7562,-46.8666,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0000,https://static.inaturalist.org/sounds/1216197....,1161364/iNat1216197.ogg,iNat
1,1161364,[],[],-22.7558,-46.8700,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0000,https://static.inaturalist.org/sounds/1114648....,1161364/iNat1114648.ogg,iNat
2,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0000,https://static.inaturalist.org/sounds/810195.m...,1161364/iNat810195.ogg,iNat
3,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0000,https://static.inaturalist.org/sounds/818781.m...,1161364/iNat818781.ogg,iNat
4,1161364,[],[],-22.7426,-46.8985,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0000,https://static.inaturalist.org/sounds/556514.m...,1161364/iNat556514.ogg,iNat


,filename,start,end,primary_label,numLabel
0,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:00,00:00:05,22961;23158;24321;517063;65380,5
1,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:05,00:00:10,22961;23158;24321;517063;65380,5
2,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:10,00:00:15,22961;23158;24321;517063;65380,5
3,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:15,00:00:20,22961;23158;24321;517063;65380,5
4,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:20,00:00:25,22961;23158;24321;517063;65380,5


In [3]:
print(texonomy_csv.shape)
print(train_csv.shape)
print(labels.shape)

(234, 5)
(35549, 15)
(1478, 5)


In [4]:
print(len(os.listdir("/kaggle/input/competitions/birdclef-2026/train_soundscapes")))
print(len(train_csv))
print(len(labels))

10658
35549
1478


In [5]:
print(len(os.listdir("/kaggle/input/competitions/birdclef-2026/train_audio")))

206


In [6]:
for id, group in labels.groupby("filename"): 
    if id=="BC2026_Train_0006_S09_20250828_000000.ogg": 
        display(group)

,filename,start,end,primary_label,numLabel
1134,BC2026_Train_0006_S09_20250828_000000.ogg,00:00:05,00:00:10,516975,1
1135,BC2026_Train_0006_S09_20250828_000000.ogg,00:00:10,00:00:15,516975,1
1136,BC2026_Train_0006_S09_20250828_000000.ogg,00:00:05,00:00:10,516975,1
1137,BC2026_Train_0006_S09_20250828_000000.ogg,00:00:10,00:00:15,516975,1


# Process the dataset

In [7]:
sampleSubCo = pd.read_csv('/kaggle/input/competitions/birdclef-2026/taxonomy.csv')['primary_label'].tolist() # list(sampleSub.columns)
# sampleSubCo.pop(0)


#combinedDataset = pd.DataFrame(columns=['fileName'] + sampleSubCo)
#combinedDataset

In [8]:
print(os.listdir("/kaggle/input/competitions/birdclef-2026/test_soundscapes"))
for x in os.listdir("/kaggle/input/competitions/birdclef-2026/train_soundscapes"): 
    if x=="BC2026_Train_0020_S22_20211104_231500": 
        print("Found in train")

['readme.txt']


In [9]:
class ModelTrainDataset(Dataset):
    def __init__(self):
        self.species_cols = sampleSubCo #list(sampleSub.columns[1:])   # drop row_id
        N = len(self.species_cols) # 234
        

        sc_labels = pd.read_csv("/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv")
        sc_labels["filename"] = ("/kaggle/input/competitions/birdclef-2026/train_soundscapes/"+ sc_labels["filename"])

        # Build multi-hot species columns — only species in primary_label = 1
        for col in self.species_cols:
            sc_labels[col] = 0.0

        for idx, row in sc_labels.iterrows():
            present = [lbl.strip() for lbl in str(row["primary_label"]).split(";")]
            for col in self.species_cols:
                if col in present:
                    sc_labels.loc[idx, col] = 1.0

        # Keep only needed columns: filename + species
        sc_labels = sc_labels[["filename", 'start', 'end', 'primary_label'] + self.species_cols]
        '''
        rows = []
        for species_folder in os.listdir("/kaggle/input/competitions/birdclef-2026/train_audio"):
            for ogg_file in os.listdir(f"/kaggle/input/competitions/birdclef-2026/train_audio/{species_folder}"):
                ogg_path = f"/kaggle/input/competitions/birdclef-2026/train_audio/{species_folder}/{ogg_file}"

                info = sf.info(ogg_path)
                duration = int(info.duration)
                # print(duration)

                for x in range(0, duration, 5): 
                    if x%5 !=0: 
                        continue
                    label_row= [ogg_path, f"00:00:{x:02d}", f"00:00:{x+5:02d}", -1] + [1.0 if col == species_folder else 0.0 for col in self.species_cols]
                    rows.append(label_row)

        audio_df = pd.DataFrame(rows, columns=["filename", 'start', 'end', 'primary_label'] + self.species_cols)
        print("len audio_df: ", audio_df.shape)
        print("len sc_labels: ", sc_labels.shape)

        combined = pd.concat([sc_labels, audio_df], axis=0, ignore_index=True)
        print("Total conbined len:", combined.shape)
        
        # combined= combined.sample(frac=1, random_state=42).reset_index(drop=True) # sc_labels
        '''
        combined = sc_labels
        self.grouped = combined.groupby("filename")
        self.group_keys = list(self.grouped.groups.keys())

    def __len__(self):
        return len(self.group_keys)

    def __getitem__(self, idx):
        file_path = self.group_keys[idx]
        group = self.grouped.get_group(file_path).reset_index(drop=True)
        group["start_sec"] = pd.to_timedelta(group["start"]).dt.total_seconds().astype(int)
        group = group.sort_values("start_sec").reset_index(drop=True)
    
        audio, rate = librosa.load(file_path, sr=16000, mono=True)
    
        audList  = []
        audLabel = []
    
        for i in range(0, len(audio), rate * 5):
            chunk = audio[i : i + rate * 5]
    
            # Pad if shorter than 5 seconds 
            if len(chunk) < rate * 5:
                chunk = np.pad(chunk, (0, rate * 5 - len(chunk)))  # cleaner than list +
            chunk_idx = i // (rate * 5)   # 0, 1, 2, 3 ...
    
            #  Stop if no more label rows 
            if chunk_idx >= len(group):
                break
    
            audList.append(chunk.astype(np.float32))                              # (80000,)
            audLabel.append(group.iloc[chunk_idx][self.species_cols]              # ← single row
                            .values.astype(np.float32))                           # (234,)
    
        return audList, audLabel


dataset = ModelTrainDataset()

In [10]:
def collate_func(batch): 
    all_cunks = []
    all_labels = []

    for ch, lab in batch: 
        for data in ch: 
            all_cunks.append(data)
        for data in lab: 
            all_labels.append(data)
    return np.array(all_cunks) , np.array(all_labels)
    # return np.concat(all_cunks, axis=0), np.concat(all_lanels, axis=0)


modelDataLoader = DataLoader(dataset , batch_size=2, shuffle=True, collate_fn=collate_func)
#for trainData , labels in modelDataLoader: 
#    print(trainData.shape, labels.shape)
    #print(trainData['filename'][0])
    #pass

# Now process the test dataset and dataloader

In [11]:
class testDataset(Dataset): 
    def __init__(self): 
        testSounds = os.listdir("/kaggle/input/competitions/birdclef-2026/test_soundscapes")
        testSounds = [x for x in testSounds if x.endswith(".ogg")]
        self.usingTest_sound = True
        
        if len(testSounds) == 0: 
            self.testSounds = [
                'BC2026_Train_0001_S08_20250606_030007.ogg',
                'BC2026_Train_0002_S08_20250607_030007.ogg',
                'BC2026_Train_0003_S08_20250607_070007.ogg',
                'BC2026_Train_0004_S08_20250607_070007.ogg',
                'BC2026_Train_0005_S08_20250607_070007.ogg',
                'BC2026_Train_0006_S09_20250828_000000.ogg',
                'BC2026_Train_0007_S09_20250829_000000.ogg',
                'BC2026_Train_0008_S09_20250831_000000.ogg',
                'BC2026_Train_0009_S09_20250828_000000.ogg',
                'BC2026_Train_0010_S09_20250828_000000.ogg',
                'BC2026_Train_0011_S03_20211114_200000.ogg',
                'BC2026_Train_0012_S03_20220211_233000.ogg',
                'BC2026_Train_0013_S13_20220217_233000.ogg',
                'BC2026_Train_0014_S13_20220228_214500.ogg',
                'BC2026_Train_0015_S18_20211016_011500.ogg',
                'BC2026_Train_0016_S18_20220204_221500.ogg',
                'BC2026_Train_0017_S22_20211026_233000.ogg',
                'BC2026_Train_0018_S22_20211028_234500.ogg',
                'BC2026_Train_0019_S22_20211104_200000.ogg',
                'BC2026_Train_0020_S22_20211104_231500.ogg'
            ]
            self.usingTest_sound = False
            
    def __len__(self): 
        return len(self.testSounds)

    def __getitem__(self, idx:int): 
        oggName = self.testSounds[idx]
        if self.usingTest_sound == True: 
            file_path = f"/kaggle/input/competitions/birdclef-2026/test_soundscapes/{oggName}" 
        else: 
            file_path = f"/kaggle/input/competitions/birdclef-2026/train_soundscapes/{oggName}" 

        
        audOggName = []
        audList = []
        for x in range(0, 60, 5): 
            audOggName.append(f"{oggName.split(".")[0]}_{str(x+5)}")
            audio, _ = librosa.load(
                file_path,
                sr = 16000,
                mono = True,
                offset = x, # start reading from this second
                duration = 5, #5 sec
            )
            audList.append(list(audio))

        return audOggName , audList


test_dataset = testDataset()

In [12]:
"BC2026_Test_0001_S05_20250227_010002" in os.listdir("/kaggle/input/competitions/birdclef-2026/train_soundscapes")

False

# Load AST model: =======>>>>> XXX <<<<=======

In [13]:
class ASTClassifier_OnlyTrainable_Head(nn.Module): 
    def __init__(self, backbone, embed_dim, n_classes): 
        super().__init__()
        self.backbone = backbone 
        
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim), 
            nn.Dropout(0.2),
            nn.Linear(embed_dim, 512), 
            nn.GELU(), 
            nn.Dropout(0.1), 
            nn.Linear(512, n_classes)
            
        )

    def forward(self, input_values): 
        out = self.backbone(input_values)
        return self.head(out.pooler_output)
        

In [14]:
class ASTClassifier_FullTrainable_Model(nn.Module): 
    def __init__(self, backbone, embed_dim, n_classes):
        
        super().__init__()
        self.backbone = backbone
        
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.2),
            nn.Linear(embed_dim, 512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, n_classes),
        )

    def forward(self, input_values):
        out = self.backbone(input_values=input_values)
        return self.head(out.pooler_output)   # pooler_output: (B, 768)

    def freeze_backbone(self): 
        for p in self.backbone.parameters(): 
            p.requires_grad = False

    def unfreeze_backbone(self): # unfreeze parametes for full fine-tuning
        for p in self.backbone.parameters: 
            p.requires_grad = True

    def unfreeze_last_n_layers(self, n=4): 
        for p in self.backbone.parameters(): 
            p.requires_grad = False

        # now unfreze last n layer + final layer Norm layer of AST
        encLayer = self.backbone.encoder.layer
        totalEncLayer = len(encLayer)

        for layer in encLayer[totalEncLayer - n:]: 
            for p in layer.parameters(): 
                p.requires_grad = True

        for p in self.backbone.layernorm.parameters(): 
            p.requires_grad = True

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in self.parameters())
        print(f"Unfroze last {n}/{totalEncLayer} encoder layers")
        print(f"Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M params  ({100*trainable/total:.1f}%)")


    
    def get_optimizer(self, strategy="differential", lr_head=1e-3, lr_backbone=1e-5):
        
        if strategy == "head_only": 
            self.freeze_backbone()
            self.torch.optim.AdamW(
                filter(lambda p: p.requires_grad, self.parameters()),
                lr=lr_head, weight_decay=1e-4
            )
            
        elif strategy == "full":
            self.unfreeze_backbone()
            return torch.optim.AdamW(
                self.parameters(), lr=lr_backbone, weight_decay=1e-4
            )

        elif strategy == "differential":
            self.unfreeze_backbone()
            # Different parameter groups → different learning rates
            return torch.optim.AdamW([
                {"params": self.backbone.parameters(), "lr": lr_backbone},  # tiny
                {"params": self.head.parameters(),     "lr": lr_head},      # large
            ], weight_decay=1e-4)

        else:
            raise ValueError(f"Unknown strategy: {strategy}")


In [15]:
def load_ast_offline(n_classes=234, freeze_backbone=False, trainMethod="outerLayerTrainable"): # loaded pretrained weights 
    AST_DIR = "/kaggle/input/datasets/niazmahmud0201/offlinebard-dataset/offline_weights/ast-audioset"

    if not Path(AST_DIR).exists(): 
        raise FileNotFoundError(f"AST pre-trained model not found in {AST_DIR}")

    print("AST model founded and start loading AST model!!!")
    
    feature_extractor = ASTFeatureExtractor.from_pretrained(AST_DIR)
    backbone_ast = ASTModel.from_pretrained(AST_DIR)


    if freeze_backbone==True: 
        for p in backbone_ast.parameters(): 
            p.require_grad = False
        print(f"AST  model backBone Frized")

    
    if trainMethod == "outerLayerTrainable": 
        model = ASTClassifier_OnlyTrainable_Head(
            backbone = backbone_ast,
            embed_dim = backbone_ast.config.hidden_size, # 768
            n_classes = n_classes # 234
        )
    else: 
        model = ASTClassifier_FullTrainable_Model(
            backbone = backbone_ast,
            embed_dim = backbone_ast.config.hidden_size, # 768
            n_classes = n_classes # 234
        )

        

    print(f"Loaded AST model with Parameter:{sum(p.numel() for p in model.parameters()) / 1e6}")
    return model , feature_extractor



device = "cuda" if torch.cuda.is_available() else "cpu"
myAST_model , AST_Feature_extractor =  load_ast_offline(n_classes=234, freeze_backbone=False, trainMethod="FullTrainableModel")
myAST_model  = nn.DataParallel(myAST_model,  device_ids=[0, 1])


myAST_model = myAST_model.to(device)

# get the raw model to get the optimizers for training 
raw_model = myAST_model.module if isinstance(myAST_model, nn.DataParallel) else myAST_model

AST model founded and start loading AST model!!!


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ASTModel LOAD REPORT from: /kaggle/input/datasets/niazmahmud0201/offlinebard-dataset/offline_weights/ast-audioset
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded AST model with Parameter:86.70257


In [16]:
# laod the audiofile.ogg file in specific sampling rate for AST model
audio_16k_1, samplingRate_1 = librosa.load("/kaggle/input/competitions/birdclef-2026/train_audio/1161364/iNat1114648.ogg", sr=16000)
audio_16k_2, samplingRate_2 = librosa.load("/kaggle/input/competitions/birdclef-2026/train_audio/1595929/XC999239.ogg", sr=16000)
print("audio_16k_1  shape: ", audio_16k_1.shape , "  | samplingRate_1:", samplingRate_1)
print("audio_16k_2  shape: ", audio_16k_2.shape , "  | samplingRate_2:", samplingRate_2)



# Now use AST's pwn feature extractor for .ogg voice  : 16 kHz = 16000 audio
inputs = AST_Feature_extractor(
    [audio_16k_1 , audio_16k_2],          # numpy array, 16 kHz
    sampling_rate=16000,
    return_tensors="pt",
    padding=True,
)

print(type(inputs))
print(type(inputs["input_values"]))

print("AST input shape: ", inputs["input_values"].shape) 
logits = myAST_model(inputs["input_values"].to(device))   # shape: (batch, 234)
ast_probab = torch.sigmoid(logits)

print("AST Logits output: ", ast_probab.shape) 

audio_16k_1  shape:  (454272,)   | samplingRate_1: 16000
audio_16k_2  shape:  (412800,)   | samplingRate_2: 16000
<class 'transformers.feature_extraction_utils.BatchFeature'>
<class 'torch.Tensor'>
AST input shape:  torch.Size([2, 1024, 128])
AST Logits output:  torch.Size([2, 234])


## Full Parameter FineTuning of "AST" model

In [17]:
#device = "cuda" if torch.cuda.is_available() else "cpu"
#myAST_model , AST_Feature_extractor =  load_ast_offline(n_classes=234, freeze_backbone=False,  trainMethod="FullParamModel")

#myAST_model = myAST_model.to(device)

In [18]:
#optimizer = myAST_model.get_optimizer("head_only", lr_head=1e-3) # head only (fastest, least memory)
#optimizer = myAST_model.get_optimizer("full", lr_backbone=1e-5) # full fine-tune (slowest, most memory ~16GB VRAM)
#optimizer = myAST_model.get_optimizer("differential", lr_head=1e-3, lr_backbone=1e-5) # differential LR (RECOMMENDED — balanced)


# partial unfreeze + differential (best for limited GPU)
raw_model.unfreeze_last_n_layers(n=4) # only last 4 of 12 transformer layers
optimizer = torch.optim.AdamW([
    {"params": raw_model.backbone.parameters(), "lr": 1e-5},
    {"params": raw_model.head.parameters(),     "lr": 1e-3},
], weight_decay=1e-4)



# scheduler (cosine warmup — works well for transformers) ───────────────────
N_EPOCHS =  4

total_steps  = len(modelDataLoader) * N_EPOCHS
warmup_steps = total_steps // 10

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

Unfroze last 4/12 encoder layers
Trainable: 28.9M / 86.7M params  (33.3%)


In [19]:
criterion = nn.BCEWithLogitsLoss()

for epoch in range(N_EPOCHS):
    myAST_model.train()

    data_bar = tqdm(modelDataLoader, desc=f"Epoch {epoch+1}/{N_EPOCHS}", unit="batch")

    running_loss = 0.0
    running_auc  = 0.0

    for batch_idx, (batch_audio, batch_labels) in enumerate(data_bar):
        # pass batched features in AST feature extractor

        inputs = AST_Feature_extractor(
            batch_audio,
            sampling_rate = 16000, 
            padding=True
        )

        input_values = torch.tensor(inputs["input_values"], dtype=torch.float32).to(device)  # (B*12, 1024, 128)
        labels = torch.tensor(batch_labels, dtype=torch.float32).to(device) # (B*12, 234)
        # print("input_values shape: ", input_values.shape , "    labels shape:", labels.shape)

        optimizer.zero_grad()
        logits = myAST_model(input_values)
        loss = criterion(logits , labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(myAST_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()



        running_loss = (running_loss * batch_idx + loss.item()) / (batch_idx + 1)

        data_bar.set_postfix({
            "loss":     f"{loss.item():.4f}",
            "avg_loss": f"{running_loss:.4f}",
            "lr":       f"{optimizer.param_groups[-1]['lr']:.2e}",
        })

    print(f"Epoch {epoch+1}/{N_EPOCHS} complete — avg_loss: {running_loss:.4f}")


Epoch 1/4: 100%|██████████| 33/33 [01:25<00:00,  2.60s/batch, loss=0.2169, avg_loss=0.2580, lr=9.32e-04]


Epoch 1/4 complete — avg_loss: 0.2580


Epoch 2/4: 100%|██████████| 33/33 [01:32<00:00,  2.81s/batch, loss=0.0724, avg_loss=0.0487, lr=5.85e-04]


Epoch 2/4 complete — avg_loss: 0.0487


Epoch 3/4: 100%|██████████| 33/33 [01:32<00:00,  2.80s/batch, loss=0.0195, avg_loss=0.0387, lr=1.78e-04]


Epoch 3/4 complete — avg_loss: 0.0387


Epoch 4/4: 100%|██████████| 33/33 [01:32<00:00,  2.80s/batch, loss=0.0527, avg_loss=0.0340, lr=0.00e+00]

Epoch 4/4 complete — avg_loss: 0.0340


# Now test and create the submission file

In [20]:
raw_model = myAST_model.module if isinstance(myAST_model, nn.DataParallel) else myAST_model

# Save trained weights to Kaggle working directory
torch.save(raw_model.state_dict(), "/kaggle/working/ast_birdclef_weights.pth")
print("Saved the model")

Saved the model
